# Company Subsidiaries via the Bigdata.com Knowledge Graph API

This notebook demonstrates how to retrieve the **full corporate subsidiary hierarchy** for a company using the Bigdata.com **Get Subsidiaries** endpoint.

**Example entity (root parent):**
- **Name:** Microsoft Corp.
- **RP Entity ID:** `228D42`

The Knowledge Graph returns every subsidiary at level 2 and deeper, including immediate parent, ultimate parent, entity name, and hierarchy depth.

**API documentation:** [Get Subsidiaries](https://docs.bigdata.com/api-reference/companies/get-subsidiaries)

---

## Prerequisites

1. A Bigdata.com API key ([developer portal](https://bigdata.com))
2. Python 3.11+ with `pandas` installed

```bash
pip install pandas jupyterlab
# or, from this repo:
cd lambda/st_agents_workflows && poetry install
```

Set your API key before running:

```bash
export BIGDATA_API_KEY="your-api-key-here"
```

In [3]:
%pip install pandas jupyterlab

  Using cached pandas-3.0.5-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached jupyterlab-4.6.3-py3-none-any.whl.metadata (16 kB)
  Using cached async_lru-2.3.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jupyter_lsp-2.3.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached jupyter_server-2.21.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached jupyterlab_server-2.28.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached notebook_shim-0.2.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.19-py3-none-any.whl.metadata (9.2 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached argon2_cffi-25.1.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached jupyter_events-0.12.1-py3-none-any.whl.me

## 1. Configuration

In [11]:
import json
import os
import urllib.error
import urllib.request
from collections import Counter

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)

API_BASE_URL = "https://api.bigdata.com"
SUBSIDIARIES_PATH = "/v1/knowledge-graph/subsidiaries/query/"
API_KEY = os.environ.get("BIGDATA_API_KEY")

# Root parent to query — change these to explore other companies.
ROOT_PARENT_ID = "228D42"
ROOT_PARENT_NAME = "Microsoft Corp."

if not API_KEY:
    raise EnvironmentError(
        "BIGDATA_API_KEY is not set. Export it before running this notebook."
    )

## 2. Call the Get Subsidiaries endpoint

**Endpoint:** `POST /v1/knowledge-graph/subsidiaries/query/`

**Request body:** pass the company's RavenPack Entity ID (`rp_entity_id`).

**Response fields (per subsidiary):**

| Field | Description |
|---|---|
| `identifier` | RP Entity ID of the subsidiary |
| `entity_name` | Name of the subsidiary |
| `level` | Depth in the hierarchy (2 = direct subsidiary; 3 = grandchild; etc.) |
| `immediate_parent_id` / `immediate_parent_name` | Direct parent entity |
| `ultimate_parent_id` / `ultimate_parent_name` | Top of the corporate tree (root parent) |

In [12]:
def fetch_subsidiaries(rp_entity_id: str) -> dict:
    """Return the full subsidiary hierarchy for a RavenPack entity ID."""
    payload = {
        "identifier": {
            "type": "rp_entity_id",
            "value": rp_entity_id,
        }
    }
    body = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        url=f"{API_BASE_URL}{SUBSIDIARIES_PATH}",
        data=body,
        method="POST",
        headers={
            "Content-Type": "application/json",
            "X-API-KEY": API_KEY,
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"API error {exc.code}: {detail}") from exc


response = fetch_subsidiaries(ROOT_PARENT_ID)
subsidiaries = response.get("results", [])
metadata = response.get("metadata", {})
errors = response.get("errors", [])

if errors:
    raise RuntimeError(f"API returned errors: {errors}")

print(f"Fetched {len(subsidiaries)} subsidiaries for {ROOT_PARENT_NAME} ({ROOT_PARENT_ID})")
print(f"Request ID: {metadata.get('request_id')}")
print(f"Timestamp:  {metadata.get('timestamp')}")

Fetched 895 subsidiaries for Microsoft Corp. (228D42)
Request ID: b613c467-144e-4e3e-b945-7ed0e99fee7f
Timestamp:  2026-09-09T18:05:12.153572+00:00


## 3. Summary statistics

In [13]:
level_counts = Counter(row["level"] for row in subsidiaries)
max_level = max(level_counts) if level_counts else None
direct_subs = [row for row in subsidiaries if row["level"] == 2]

summary_rows = [
    {"Metric": "Root parent", "Value": f"{ROOT_PARENT_NAME} ({ROOT_PARENT_ID})"},
    {"Metric": "Total subsidiaries returned", "Value": len(subsidiaries)},
    {"Metric": "Direct subsidiaries (level 2)", "Value": len(direct_subs)},
    {"Metric": "Deepest level in tree", "Value": max_level},
    {"Metric": "API request ID", "Value": metadata.get("request_id", "—")},
    {"Metric": "Retrieved at (UTC)", "Value": metadata.get("timestamp", "—")},
]

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

if level_counts:
    level_breakdown = pd.DataFrame(
        [
            {"Level": level, "Count": level_counts[level]}
            for level in sorted(level_counts)
        ]
    )
    display(Markdown("**Hierarchy depth breakdown**"))
    display(level_breakdown)

,Metric,Value
0,Root parent,Microsoft Corp. (228D42)
1,Total subsidiaries returned,895
2,Direct subsidiaries (level 2),420
3,Deepest level in tree,7
4,API request ID,b613c467-144e-4e3e-b945-7ed0e99fee7f
5,Retrieved at (UTC),2026-09-09T18:05:12.153572+00:00


**Hierarchy depth breakdown**

,Level,Count
0,2,420
1,3,354
2,4,105
3,5,12
4,6,2
5,7,2


## 4. Direct subsidiaries (level 2)

These are entities owned directly by the root parent.

In [14]:
direct_df = pd.DataFrame(direct_subs)[
    ["identifier", "entity_name", "level", "immediate_parent_name"]
].rename(
    columns={
        "identifier": "Entity ID",
        "entity_name": "Entity Name",
        "level": "Level",
        "immediate_parent_name": "Immediate Parent",
    }
)

display(direct_df if not direct_df.empty else pd.DataFrame([{"Note": "No direct subsidiaries found."}]))

,Entity ID,Entity Name,Level,Immediate Parent
0,03F4FE,Navision A/S,2,Microsoft Corp.
1,048590,Activision Blizzard Inc.,2,Microsoft Corp.
2,051799,Deis Inc.,2,Microsoft Corp.
3,052F44,AltspaceVR Inc.,2,Microsoft Corp.
4,05I4UQ,Microsoft Mobile Telecommunications Ltd.,2,Microsoft Corp.
...,...,...,...,...
415,W3L7AS,Microsoft Mobile Technology Shenzhen Co. Ltd.,2,Microsoft Corp.
416,W6KTZP,Microsoft De Argentina SA,2,Microsoft Corp.
417,WJMPQS,Microsoft Venezuela SA,2,Microsoft Corp.
418,WKEQ9Q,Microsoft Mobile (China) Investments Co. Ltd.,2,Microsoft Corp.


## 5. Sample subsidiaries (first 10 by level, then name)

Preview of the corporate tree below the root parent.

In [15]:
DISPLAY_COLUMNS = [
    "level",
    "identifier",
    "entity_name",
    "immediate_parent_name",
]

sorted_subsidiaries = sorted(
    subsidiaries,
    key=lambda row: (row["level"], (row.get("entity_name") or "").lower()),
)

sample_df = pd.DataFrame(sorted_subsidiaries[:10])[DISPLAY_COLUMNS].rename(
    columns={
        "level": "Level",
        "identifier": "Entity ID",
        "entity_name": "Entity Name",
        "immediate_parent_name": "Immediate Parent",
    }
)

display(sample_df)

,Level,Entity ID,Entity Name,Immediate Parent
0,2,D849EE,3DV Systems Ltd.,Microsoft Corp.
1,2,617BA9,6 Wunderkinder GmbH,Microsoft Corp.
2,2,81C81C,90 Degree Software Inc.,Microsoft Corp.
3,2,130GBE,Access Software Inc.,Microsoft Corp.
4,2,9C7BC9,Acompli Inc.,Microsoft Corp.
5,2,048590,Activision Blizzard Inc.,Microsoft Corp.
6,2,F1F4A8,Adallom Inc.,Microsoft Corp.
7,2,MULYT9,ADRM Software Inc.,Microsoft Corp.
8,2,8C63EF,Adxstudio Inc.,Microsoft Corp.
9,2,OAFSY3,Affirmed Networks Inc.,Microsoft Corp.


## 6. Complete subsidiary table

Full list sorted by hierarchy level, then entity name.

In [16]:
full_df = pd.DataFrame(sorted_subsidiaries).rename(
    columns={
        "level": "Level",
        "identifier": "Entity ID",
        "entity_name": "Entity Name",
        "immediate_parent_id": "Immediate Parent ID",
        "immediate_parent_name": "Immediate Parent",
        "ultimate_parent_id": "Ultimate Parent ID",
        "ultimate_parent_name": "Ultimate Parent",
    }
)[
    [
        "Level",
        "Entity ID",
        "Entity Name",
        "Immediate Parent ID",
        "Immediate Parent",
        "Ultimate Parent ID",
        "Ultimate Parent",
    ]
]

display(full_df)
print(f"\nTotal rows: {len(full_df)}")

,Level,Entity ID,Entity Name,Immediate Parent ID,Immediate Parent,Ultimate Parent ID,Ultimate Parent
0,2,D849EE,3DV Systems Ltd.,228D42,Microsoft Corp.,228D42,Microsoft Corp.
1,2,617BA9,6 Wunderkinder GmbH,228D42,Microsoft Corp.,228D42,Microsoft Corp.
2,2,81C81C,90 Degree Software Inc.,228D42,Microsoft Corp.,228D42,Microsoft Corp.
3,2,130GBE,Access Software Inc.,228D42,Microsoft Corp.,228D42,Microsoft Corp.
4,2,9C7BC9,Acompli Inc.,228D42,Microsoft Corp.,228D42,Microsoft Corp.
...,...,...,...,...,...,...,...
890,5,EY6DB8,Peltarion Energy AB,PR4FG3,Peltarion AB,228D42,Microsoft Corp.
891,6,DPDHH2,Midasplayer (Skills) Ltd.,JZRKV2,Midasplayer Malta Holding Co. Ltd.,228D42,Microsoft Corp.
892,6,QNKF3C,Midasplayer.com Ltd.,JZRKV2,Midasplayer Malta Holding Co. Ltd.,228D42,Microsoft Corp.
893,7,WBGSJA,Fabrication Games Europe AB,QNKF3C,Midasplayer.com Ltd.,228D42,Microsoft Corp.



Total rows: 895


## 7. Export (optional)

Save the full hierarchy to CSV or JSON for downstream use.

In [17]:
OUTPUT_STEM = f"{ROOT_PARENT_ID.lower()}_subsidiaries"
csv_path = f"{OUTPUT_STEM}.csv"
json_path = f"{OUTPUT_STEM}.json"

full_df.to_csv(csv_path, index=False)
with open(json_path, "w", encoding="utf-8") as handle:
    json.dump({"metadata": metadata, "results": sorted_subsidiaries}, handle, indent=2)

print(f"Wrote {csv_path} ({len(full_df)} rows)")
print(f"Wrote {json_path}")

Wrote 228d42_subsidiaries.csv (895 rows)
Wrote 228d42_subsidiaries.json
